In [1]:
#!/usr/bin/env julia
"""
collect_exercises.jl

Collects all exercises from lecture notebooks (L##.ipynb) and writes them
to a single Quarto .qmd output file, using .exercise-box div syntax
compatible with random-2.ipynb.

Skips L00.ipynb.

Quizzes: every `Q<n>.qmd` in exercises/quizzes is inserted after the lecture given by
its `after-lecture:` front matter field. Each top-level `::: {.quiz-question}` block
becomes an exercise labelled "Quiz <n>: Exercise <k>" whose solution is available in
the bank immediately (`release="now"`).
Embeds all code cell outputs (plots, text, HTML) as inline content.

Usage:
    julia collect_exercises.jl
"""

using JSON
using Base64

# ── Configuration ─────────────────────────────────────────────────────────────
lectures_dir = "../lectures"
quizzes_dir  = "../exercises/quizzes"
output_file  = "../exercises/exercises.qmd"

# ── Helpers ───────────────────────────────────────────────────────────────────

"""
    cell_to_markdown(cell, lang) -> String

Convert a single notebook cell to a Markdown string.
- Markdown cells: returned as-is.
- Code cells: source in a collapsible fenced block, followed by all outputs:
  image/png and image/svg+xml as base64 img, text/html as raw HTML,
  text/plain and stream as fenced code blocks.
"""
function cell_to_markdown(cell, lang::String="julia")::String
    buf = IOBuffer()
    src = join(cell["source"])
    isempty(strip(src)) && return ""

    if cell["cell_type"] == "markdown"
        println(buf, src)
    else  # code / raw
        # drop Quarto cell options (`#| code-fold: true`, ...): they mean nothing in the bank
        code = join(filter(l -> !startswith(lstrip(l), "#|"), split(src, '\n')), '\n')
        code = strip(code, '\n')
        println(buf, "<details><summary>Code</summary>\n")
        println(buf, "```$lang")
        println(buf, code)
        println(buf, "```")
        println(buf, "</details>\n")

        # Embed all outputs produced by this cell
        for output in get(cell, "outputs", [])
            data = get(output, "data", Dict())

            if haskey(data, "image/png")
                b64 = replace(string(data["image/png"]), '\n' => "")
                println(buf, "\n<img src=\"data:image/png;base64,$b64\" style=\"max-width:100%;height:auto;display:block;margin:.5em 0;\">")

            elseif haskey(data, "image/svg+xml")
                # embed as an image (inlining the svg itself can clash with other svgs' ids)
                svg = join(data["image/svg+xml"])
                println(buf, "\n<img src=\"data:image/svg+xml;base64,$(base64encode(svg))\" style=\"max-width:100%;height:auto;display:block;margin:.5em 0;\">")

            elseif haskey(data, "text/html")
                html = join(data["text/html"])
                println(buf, "\n$html")

            elseif haskey(data, "text/plain")
                txt = join(data["text/plain"])
                println(buf, "\n```\n$txt\n```")

            elseif get(output, "output_type", "") == "stream"
                txt = join(get(output, "text", []))
                isempty(strip(txt)) || println(buf, "\n```\n$txt\n```")
            end
        end
    end

    return String(take!(buf))
end

"""
    notebook_to_markdown(path) -> String

Convert a Jupyter notebook to a single Markdown string.
"""
function notebook_to_markdown(path::String)::String
    nb   = JSON.parsefile(path)
    lang = get(get(get(nb, "metadata", Dict()), "kernelspec", Dict()), "language", "julia")
    buf  = IOBuffer()
    for cell in nb["cells"]
        md = cell_to_markdown(cell, lang)
        isempty(md) && continue
        println(buf, md)
    end
    return String(take!(buf))
end

"""
    extract_exercises(markdown, lecture_label) -> Vector{String}

Pull every `::: {#exr-...}` block out of markdown and re-wrap as
`::: {.exercise-box #bank-exr-...}` with a bold cross-reference label.
Extra attributes are kept, e.g. `::: {#exr-1 .no-solution}` marks an exercise
that doesn't need a solution (it is not counted as missing in the exercise report).
Nested `:::` blocks (hints, solutions) are preserved.
Trailing outputs immediately after closing ::: are pulled into the block.
"""
function extract_exercises(markdown::String, lecture_label::String)::Vector{String}
    exercises = String[]
    lines     = split(markdown, '\n')
    n         = length(lines)
    i         = 1

    while i <= n
        line = lines[i]
        m = match(r"^:::\s*\{#(exr-[\w-]+)([^}]*)\}", line)
        if m !== nothing
            exr_id = m.captures[1]
            extra  = strip(m.captures[2])   # e.g. `.no-solution` (exercise doesn't need a solution)
            depth  = 1
            i     += 1

            # Collect inner lines until matching closing :::
            inner = String[]
            while i <= n && depth > 0
                l = lines[i]
                if occursin(r"^:::\s*\{", l)
                    depth += 1
                    push!(inner, l)
                elseif strip(l) == ":::"
                    depth -= 1
                    depth > 0 && push!(inner, l)
                else
                    push!(inner, l)
                end
                i += 1
            end

            # Remove bank/practice links (redundant inside the bank)
            inner = filter(l -> !occursin("bank.html", l) && !occursin("random-2.html", l), inner)

            # Strip leading/trailing blank lines
            while !isempty(inner) && isempty(strip(first(inner))); popfirst!(inner); end
            while !isempty(inner) && isempty(strip(last(inner)));  pop!(inner);      end

            # Remove equation numbers
            inner = map(inner) do line
                line = replace(line, r"\s*\{#eq-[\w-]+\}" => "")
                line = replace(line, r"\\tag\{[^}]*\}"    => "")
                line
            end

            # Peek ahead: grab outputs immediately following closing :::
            trailing = String[]
            j = i
            while j <= n
                l = strip(lines[j])
                if isempty(l)
                    j += 1
                elseif startswith(l, "<img ") || (startswith(l, "<") && !startswith(l, "<details>") && !startswith(l, "</details>"))
                    push!(trailing, lines[j])
                    j += 1
                elseif l == "```"
                    push!(trailing, lines[j]); j += 1
                    while j <= n && strip(lines[j]) != "```"
                        push!(trailing, lines[j]); j += 1
                    end
                    j <= n && (push!(trailing, lines[j]); j += 1)
                else
                    break
                end
            end
            if !isempty(trailing)
                i = j
                append!(inner, ["", trailing...])
            end

            block = "<!-- $lecture_label -->\n" *
                    "::: {.exercise-box #bank-$exr_id$(isempty(extra) ? "" : " " * extra)}\n" *
                    "**@$exr_id**\n\n" *
                    join(inner, '\n') * "\n" *
                    ":::"

            push!(exercises, block)
        else
            i += 1
        end
    end
    return exercises
end

"""
    read_quiz(path) -> (number, after_lecture, blocks)

Read a quiz file `Q<n>.qmd` and turn each top-level `::: {.quiz-question}` block
into an exercise-box labelled "Quiz <n>: Exercise <k>" (solutions released immediately).
"""
function read_quiz(path::String)
    quiz_num = parse(Int, match(r"^Q(\d+)\.qmd$", basename(path)).captures[1])
    text     = read(path, String)

    fm    = match(r"^---\n(.*?)\n---\n"s, text)
    after = fm === nothing ? nothing : match(r"^after-lecture:\s*(\d+)"m, fm.captures[1])
    after === nothing && error("$(basename(path)): missing `after-lecture:` in the front matter")
    after_lecture = parse(Int, after.captures[1])
    body = fm === nothing ? text : text[length(fm.match)+1:end]
    body = replace(body, r"<!--.*?-->"s => "")      # drop comments

    blocks = String[]
    lines  = split(body, '\n')
    i      = 1
    while i <= length(lines)
        if occursin(r"^:::\s*\{\s*\.quiz-question", lines[i])
            depth, inner = 1, String[]
            i += 1
            while i <= length(lines) && depth > 0
                l = lines[i]
                if occursin(r"^:::\s*\{", l)
                    depth += 1
                elseif strip(l) == ":::"
                    depth -= 1
                end
                depth > 0 && push!(inner, l)
                i += 1
            end
            while !isempty(inner) && isempty(strip(first(inner))); popfirst!(inner); end
            while !isempty(inner) && isempty(strip(last(inner)));  pop!(inner);      end

            k = length(blocks) + 1
            push!(blocks, "<!-- Q$quiz_num -->\n" *
                          "::: {.exercise-box #bank-q$quiz_num-$k release=\"now\"}\n" *
                          "**Quiz $quiz_num: Exercise $k**\n\n" *
                          join(inner, '\n') * "\n" *
                          ":::")
        else
            i += 1
        end
    end
    return quiz_num, after_lecture, blocks
end

"""
    write_quizzes(io, quizzes)

Write each quiz as a `## Quiz <n>` section.
"""
function write_quizzes(io, quizzes)
    for (quiz_num, _, blocks) in quizzes
        println("  Q$quiz_num: $(length(blocks)) question(s)")
        println(io, "## Quiz $quiz_num\n")
        for b in blocks
            println(io, b)
            println(io)
        end
    end
end

# ── Main ──────────────────────────────────────────────────────────────────────

function main()
    pattern   = r"^L\d+\.ipynb$"
    all_files = filter(f -> occursin(pattern, f), readdir(lectures_dir))
    sort!(all_files)
    all_files = filter(f -> f != "L00.ipynb", all_files)

    if isempty(all_files)
        @warn "No lecture notebooks found in $lectures_dir"
        return
    end

    println("Found $(length(all_files)) lecture notebook(s): $(join(all_files, ", "))")

    quiz_files = isdir(quizzes_dir) ? filter(f -> occursin(r"^Q\d+\.qmd$", f), readdir(quizzes_dir)) : String[]
    quizzes    = sort([read_quiz(joinpath(quizzes_dir, f)) for f in quiz_files])
    println("Found $(length(quizzes)) quiz file(s)")
    written    = Set{Int}()

    mkpath(dirname(output_file))
    open(output_file, "w") do io
        write(io, """
---
format:
  html:
    code-copy: true
    toc: true
---

""")

        for fname in all_files
            label   = splitext(fname)[1]
            num_str = replace(label, r"^L0*" => "")
            lec_num = tryparse(Int, num_str)

            path      = joinpath(lectures_dir, fname)
            markdown  = notebook_to_markdown(path)
            exercises = extract_exercises(markdown, label)

            if isempty(exercises)
                println("  $label: no exercises found")
            else
                println("  $label: $(length(exercises)) exercise(s)")
                println(io, "## Lecture $lec_num\n")
                for ex in exercises
                    println(io, ex)
                    println(io)
                end
            end

            # quizzes that come after this lecture
            after_this = filter(q -> q[2] == lec_num, quizzes)
            write_quizzes(io, after_this)
            union!(written, first.(after_this))
        end

        # quizzes after a lecture that doesn't exist (yet) go at the end
        write_quizzes(io, filter(q -> !(q[1] in written), quizzes))
    end

    println("\nWrote exercises to: $output_file")
end

main()


Found 5 lecture notebook(s): L01.ipynb, L02.ipynb, L03.ipynb, L04.ipynb, L05.ipynb
Found 2 quiz file(s)
  L01: 6 exercise(s)
  L02: 9 exercise(s)
  Q0: 2 question(s)
  L03: 10 exercise(s)
  L04: 6 exercise(s)
  Q1: 4 question(s)
  L05: 6 exercise(s)

Wrote exercises to: ../exercises/exercises.qmd
